In [1]:
# So, let us dive right in. For convenience, let's begin by enabling
# automatic reloading of modules when they change.
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from collections import OrderedDict
import pyEPR as epr



## Design and GUI Startup

In [ ]:
design = designs.DesignPlanar()
design.overwrite_enabled = True

try:
    gui.main_window.force_close = True
    gui.main_window.close()
except (NameError, AttributeError):
    pass

gui = MetalGUI(design)

12:48PM 26s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).
12:48PM 26s INFO [_start_renderers]: Renderer=aedt_q3d skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').
12:48PM 26s INFO [_start_renderers]: Renderer=aedt_hfss skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').


12:49PM 09s WARNING [_qt_message_handler]: WARNING: Could not create pixmap from c:\Users\thoma\miniconda3nvs\qm-env\Lib\site-packages\qiskit_metal\_gui\styles\metal_dark\rc\transparent.png (No context available from Qt)
Python Traceback (most recent call last):
  File "c:\Users\thoma\miniconda3\envs\qm-env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\thoma\miniconda3\envs\qm-env\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\Users\thoma\miniconda3\envs\qm-env\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:\Users\thoma\miniconda3\envs\qm-env\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "c:\Users\thoma\miniconda3\envs\qm-env\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "c:\Users\thoma\miniconda3\envs\qm-env\Lib\

## Design 

Parameters

In [4]:
design.chips.main.size.size_x = '5mm'
design.chips.main.size.size_y = '5mm'
design.chips.main.size.size_z = '-380um'
design.chips.main.size.sample_holder_top='2mm'
design.chips.main.size.sample_holder_bottom='380um'

lp_width = '180um'
lp_gap = '140um'

tl_width = '15.5um'
tl_gap = '9um'

rr_width = tl_width
rr_gap = tl_gap
rr_coupling_gap = '15um' 
rr_coupling_length = '150um'
rr_termination = rr_gap


Launch Pads

In [5]:

LP1 = LaunchpadWirebondDriven(design,name='LP1',
                       options={'orientation': '0', 'pos_x': '-2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap ,'pad_height':'180um', 'trace_width': tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )




LP2 = LaunchpadWirebondDriven(design,name='LP2',
                       options={'orientation': '180', 'pos_x': '2.1mm', 'pos_y': '0mm','pad_width':lp_width ,'pad_gap':lp_gap, 'pad_height':'180um', 'trace_width':tl_width , 'trace_gap': tl_gap},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

gui.rebuild()
gui.autoscale()

Resonator 1

In [6]:
OTG1 = OpenToGround(design,name='OTG1'
                 ,options={'pos_x':'-1.5mm','pos_y':'-1.4mm','orientation':'-90','width':rr_width, 'gap':rr_gap, 'termination_gap': rr_termination })


q_read1 = CoupledLineTee(design,'Q_Read1', options=dict(pos_x = '-1.7mm', pos_y = '0mm',
                                                        prime_width=tl_width,
                                                        prime_gap=tl_gap,
                                                        second_width=rr_width,
                                                        second_gap=rr_gap,
                                                        fillet='50um',
                                                        mirror=True,
                                                        orientation = '0',
                                                        coupling_space = rr_coupling_gap,                                                         
                                                        coupling_length = rr_coupling_length,
                                                        open_termination = False))

RR1 = RouteMeander(design, 'RR1',  Dict(
        trace_width =tl_width,
        trace_gap =tl_gap,
        total_length='3mm',
        hfss_wire_bonds = False,
        fillet='50 um',
        lead = dict(start_straight='100um',end_straight='10um'),
        meander= dict(spacing='150um',asymmetry='200um'),
        pin_inputs=Dict(
            start_pin=Dict(component='Q_Read1', pin='second_end'),
            end_pin=Dict(component='OTG1', pin='open')), ))

gui.rebuild()
gui.autoscale()

Transmission Line Segments

In [7]:
seg1 = RoutePathfinder(design, 'seg1', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='LP1',
                                                                pin='tie'),
                                                       end_pin=dict(
                                                        component='Q_Read1',
                                                        pin='prime_start')
                                                 )))



seg2 = RoutePathfinder(design, 'seg2', options = dict(chip='main', 
                                                      trace_width =tl_width,
                                                      trace_gap =tl_gap,
                                                      fillet='100um',
                                                      hfss_wire_bonds = True,
                                                    lead=dict(start_straight = '0mm',
                                                        end_straight='0mm'),
                                                        pin_inputs=dict(
                                                        start_pin=dict(
                                                                component='Q_Read1',
                                                                pin='prime_end'),
                                                       end_pin=dict(
                                                        component='LP2',
                                                        pin='tie')
                                                 )))

gui.rebuild()
gui.autoscale()

## HFSS Export

In [8]:
# pinfo = epr.ProjectInfo(
#     project_path=r'C:\Users\labuser\Documents\Ansoft\Thomas',
#     project_name='bandpass_CPW',
#     design_name='hanger_resonator_test4'
# )

# hfss = design.renderers.hfss
# hfss.start()
# hfss.render_design()